# M0 synthetic-fact knowledge audit
Run on a Kaggle GPU kernel after pushing the candidate dataset and audit code. This collects 1,260 predictions covering all 180 F/R/N_rel units using non-final prompts. It preserves matches for manual review; it never edits the candidate or trains an adapter.
The public pinned Qwen model requires no HF_TOKEN. Keep this notebook private and retain the output files before stopping the runtime.

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

# This path exists only in the remote Kaggle runtime.
remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle remote kernel first.'
repo_dir = Path(tempfile.mkdtemp(prefix='m0-audit-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main',
    'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('configs/evaluation/m0_knowledge_audit.json').is_file(), 'Push the audit files first.'

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
    '-r', 'requirements/m0-smoke.txt'], check=True)

In [ ]:
import json

# Plan-only verifies the candidate without loading any model.
plan_dir = Path(tempfile.mkdtemp(prefix='m0-audit-plan-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.m0_knowledge_audit',
    '--config', 'configs/evaluation/m0_knowledge_audit.json',
    '--output-dir', str(plan_dir), '--plan-only'], check=True)
plan = json.loads((plan_dir / 'plan.json').read_text())
assert plan['planned_predictions'] == 1260
assert plan['knowledge_units'] == 180
print('Candidate checksums verified; ready for GPU inference.')

In [ ]:
# A new directory preserves any previous audit run.
output_dir = Path(tempfile.mkdtemp(prefix='m0-knowledge-audit-', dir=remote_root))
print('Audit outputs:', output_dir, flush=True)
subprocess.run([sys.executable, '-u', '-m', 'src.m0_knowledge_audit',
    '--config', 'configs/evaluation/m0_knowledge_audit.json',
    '--output-dir', str(output_dir)], check=True)

In [ ]:
summary = json.loads((output_dir / 'summary.json').read_text())
assert summary['execution_complete']
assert summary['predictions'] == 1260
assert summary['model_executed']
print(json.dumps({key: value for key, value in summary.items() if key != 'plan'}, indent=2))
print('Preserve summary.json, predictions.jsonl, matches_for_review.json, plan.json, environment.freeze.txt')
print('Saved output directory:', output_dir)

Review all flagged matches before deciding what the model may already know. Year matches can occur by chance. A single match, including a name or title, does not prove memorization. Failed answers do not prove absence of knowledge. Preserve this baseline and the candidate unchanged. Dataset acceptance and full manual collision review are separate decisions.